# 推理 Infra 学习实录 04：KV Cache Block 的分配、复用与回收

> 归档状态（GitHub 发布副本）：源码学习稿，未做本轮 GPU 验证；block 模拟不等于模型运行验证。 所有 ModelScope 发布 URL 尚未知，不表示已发布。
> [配套文章](article.md) · [上一篇](../03/article.md) · [下一篇](../05/article.md)。Notebook 输出与执行计数已清空。
> 版本边界：nano-vLLM 安装与源码说明对齐已核对 commit `bb823b3e06983d71485a8e1f23715ebd87d98ef8`；已有 checkout 不自动覆盖，版本不符时应另选空目录。
> 路径说明：`/mnt/workspace` 是 ModelScope 平台缓存示例，请按实际环境调整，模型目录以 `snapshot_download()` 返回值为准。

这份 Notebook 用源码中的真实 `BlockManager` 做一组小型实验，观察 KV block 的分配、共享、释放和 Prefix Cache 命中。

本篇重点验证：

1. `Sequence.block_table` 保存什么。
2. 完整 block 和 partial block 的缓存差异。
3. 相同前缀如何共享物理 block。
4. 链式 hash 为什么能阻止局部相同 block 被误复用。
5. 为什么活跃 block 数比 CUDA 总显存更适合观察 nano-vLLM 的 KV 使用情况。

## 1. 环境检查

本节检查 Notebook 的 GPU 和 Python 环境。核心 block 实验不执行模型 forward，后续 Qwen 配置检查沿用本系列的 ModelScope 模型。

In [ ]:
!nvidia-smi

In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. 安装依赖

在 Jupyter 中使用 `%pip`，确保依赖安装到当前 kernel。已经安装过时可以跳过。

In [ ]:
%pip install -q modelscope pandas xxhash
%pip install -q git+https://github.com/GeeeekExplorer/nano-vllm.git@bb823b3e06983d71485a8e1f23715ebd87d98ef8

## 3. 导入源码对象

实验直接使用 nano-vLLM 的 `BlockManager` 和 `Sequence`。

In [ ]:
import pandas as pd
from IPython.display import display

from nanovllm import SamplingParams
from nanovllm.engine.block_manager import BlockManager
from nanovllm.engine.sequence import Sequence

print("BlockManager imported:", BlockManager)
print("Sequence imported:", Sequence)

## 4. 准备状态观察函数

正式配置默认每个 KV block 容纳 256 个 token。实验将 block size 缩小到 4，使边界更加直观。

In [ ]:
Sequence.block_size = 4

def block_rows(manager):
    rows = []
    for block in manager.blocks:
        rows.append({
            "block_id": block.block_id,
            "state": "used" if block.block_id in manager.used_block_ids else "free",
            "ref_count": block.ref_count,
            "has_hash": block.hash != -1,
            "token_ids": block.token_ids,
        })
    return pd.DataFrame(rows)

def show_manager(manager, *named_sequences):
    print("free_block_ids:", list(manager.free_block_ids))
    print("used_block_ids:", sorted(manager.used_block_ids))
    for name, seq in named_sequences:
        print(f"{name}.block_table:", seq.block_table)
        print(f"{name}.num_cached_tokens:", seq.num_cached_tokens)
    display(block_rows(manager))

## 5. 实验一：10 个 token 如何分成 3 个 block

10 个 token 按 block size 4 切分为 `[0..3]`、`[4..7]` 和 partial block `[8, 9]`。

In [ ]:
manager = BlockManager(num_blocks=8, block_size=4)
seq_a = Sequence(list(range(10)), SamplingParams(max_tokens=4))

cached_a = manager.can_allocate(seq_a)
manager.allocate(seq_a, cached_a)

print("seq_a.num_blocks:", seq_a.num_blocks)
print("prefix cached before first prefill:", cached_a)
for i in range(seq_a.num_blocks):
    print(f"logical block {i}:", seq_a.block(i))
show_manager(manager, ("seq_a", seq_a))

## 6. 模拟 prefill 完成并登记 Prefix Cache

`hash_blocks()` 只会登记本轮跨过的完整 block 边界。第三个 partial block 不会进入 Prefix Cache。

这不是 prefill 专属路径：每轮模型执行后由 `Scheduler.postprocess()` 调用；decode 写满新的 block 后也会登记完整块。这里仅模拟 prefill 的状态变化，不计算真实 K/V。

In [ ]:
seq_a.num_scheduled_tokens = len(seq_a)
manager.hash_blocks(seq_a)
seq_a.num_cached_tokens += seq_a.num_scheduled_tokens
seq_a.num_scheduled_tokens = 0

print("hashed token blocks:")
display(block_rows(manager))
print("hash index size:", len(manager.hash_to_block_id))

## 7. 实验二：相同前缀共享物理 block

`seq_b` 与 `seq_a` 共享前 8 个 token，只修改最后两个 token。预期命中两个前缀 block，并为尾部单独分配一个 block。

In [ ]:
seq_b = Sequence(
    [0, 1, 2, 3, 4, 5, 6, 7, 100, 101],
    SamplingParams(max_tokens=4),
)

cached_b = manager.can_allocate(seq_b)
manager.allocate(seq_b, cached_b)

print("seq_b cached blocks:", cached_b)
print("shared physical blocks:", seq_b.block_table[:cached_b])
show_manager(manager, ("seq_a", seq_a), ("seq_b", seq_b))

观察共享 block 的 `ref_count`。只要还有一条 Sequence 引用它，block 就不能回到 free list。

In [ ]:
shared_ids = list(seq_b.block_table[:cached_b])
state_df = block_rows(manager)
display(state_df.loc[state_df["block_id"].isin(shared_ids)])

## 8. 实验三：局部 block 相同，前缀不同

`seq_c` 的第二个 block 仍是 `[4, 5, 6, 7]`，但第一个 block 不同。链式 hash 应阻止它从中间开始命中。

In [ ]:
seq_c = Sequence(
    [50, 51, 52, 53, 4, 5, 6, 7, 200, 201],
    SamplingParams(max_tokens=4),
)

cached_c = manager.can_allocate(seq_c)
print("seq_c cached blocks:", cached_c)
assert cached_c == 0, "不同的完整前缀不应该命中"

## 9. 实验四：释放与引用计数

释放顺序为 `seq_a`、`seq_b`。释放 `seq_a` 后，共享 block 的引用数从 2 变成 1，并继续被 `seq_b` 使用；释放 `seq_b` 后，引用数归零。

In [ ]:
manager.deallocate(seq_a)
print("after deallocating seq_a:")
state_df = block_rows(manager)
display(state_df.loc[state_df["block_id"].isin(shared_ids)])

manager.deallocate(seq_b)
print("after deallocating seq_b:")
state_df = block_rows(manager)
display(state_df.loc[state_df["block_id"].isin(shared_ids)])
print("free_block_ids:", list(manager.free_block_ids))

释放后，block 的 hash 和 token 仍保留。只要物理 block 还没有被新内容覆盖，后续相同前缀仍有机会重新命中。

In [ ]:
seq_d = Sequence(
    [0, 1, 2, 3, 4, 5, 6, 7, 300, 301],
    SamplingParams(max_tokens=4),
)
cached_d = manager.can_allocate(seq_d)
print("cached blocks after previous requests ended:", cached_d)
assert cached_d == 2

## 10. 实验五：decode 为什么在余数为 1 时追加 block

prefill 完成后，第一个 sampled token 已 append 到 Sequence，但它的 KV 要到下一轮 decode 才写入。若它落在新 block 的第一个位置，`len(seq) % block_size == 1`。

In [ ]:
decode_manager = BlockManager(num_blocks=4, block_size=4)
decode_seq = Sequence([10, 11, 12, 13], SamplingParams(max_tokens=4))
decode_manager.allocate(decode_seq, decode_manager.can_allocate(decode_seq))

decode_seq.num_scheduled_tokens = 4
decode_manager.hash_blocks(decode_seq)
decode_seq.num_cached_tokens = 4
decode_seq.num_scheduled_tokens = 0

decode_seq.append_token(99)  # prefill 后 sample 出来的第一个 token
print("length before first decode:", len(decode_seq))
print("length % block_size:", len(decode_seq) % 4)
print("block table before may_append:", decode_seq.block_table)
print("can append:", decode_manager.can_append(decode_seq))
decode_manager.may_append(decode_seq)
print("block table after may_append:", decode_seq.block_table)

## 11. 用 Qwen3-0.6B 配置估算单个 KV block

本节只下载并读取模型配置，不初始化第二套 GPU runtime。nano-vLLM 中单个物理 KV block 的字节数为：

```text
2 × num_layers × block_size × num_kv_heads_per_rank × head_dim × dtype_bytes
```

In [ ]:
from modelscope import snapshot_download
from transformers import AutoConfig

model_dir = snapshot_download(
    "Qwen/Qwen3-0.6B",
    cache_dir="/mnt/workspace/.cache/modelscope",
)
hf_config = AutoConfig.from_pretrained(model_dir)

block_size = 256
tensor_parallel_size = 1
num_kv_heads_per_rank = hf_config.num_key_value_heads // tensor_parallel_size
head_dim = getattr(
    hf_config,
    "head_dim",
    hf_config.hidden_size // hf_config.num_attention_heads,
)
dtype_bytes = 2  # Qwen3-0.6B 常用 BF16/FP16 推理；按实际 dtype 调整

block_bytes = (
    2
    * hf_config.num_hidden_layers
    * block_size
    * num_kv_heads_per_rank
    * head_dim
    * dtype_bytes
)

print("model_dir:", model_dir)
print("layers:", hf_config.num_hidden_layers)
print("KV heads per rank:", num_kv_heads_per_rank)
print("head_dim:", head_dim)
print("block size:", block_size)
print("bytes per KV block:", block_bytes)
print("MiB per KV block:", block_bytes / 1024**2)

## 12. 为什么不只看 CUDA 总显存

nano-vLLM 的 `ModelRunner.allocate_kv_cache()` 会在初始化时按 `gpu_memory_utilization` 一次性创建 KV Cache tensor。因此，同一个引擎内改变 prompt 长度时，CUDA 已分配显存不一定跟活跃请求的 block 数同步变化。

本篇更可靠的观察指标是：

- `free_block_ids` / `used_block_ids`
- `Sequence.block_table`
- block `ref_count`
- `num_cached_tokens`
- `can_allocate()` 返回的命中 block 数

后续有真实 benchmark 时，再比较 Prefix Cache 命中前后的 prefill token 数、TTFT 或耗时。

## 13. 本篇结论

```text
Sequence token
-> 切成逻辑 block
-> BlockManager 分配或复用物理 block
-> block_table 保存映射
-> ModelRunner 计算 slot_mapping
-> Attention 写入或读取 KV Cache
-> 请求结束后按 ref_count 释放
```

下一篇将对比 nano-vLLM 的 block Prefix Cache 和 mini-SGLang 的 Radix Cache。